<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/Python%E7%A8%8B%E5%BC%8F%E8%AA%9E%E8%A8%80_(III)_%EF%BC%93.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Python程式語言 (III)-3**

> 前幾章的頻域轉換法，可以被一種強大的時域濾波器給取代，不需要再浪費力氣轉到頻域了！

> 在學術上，肌電訊號的頻率能量是從20Hz到400Hz。

> 在施力的過程，手臂或腿部的晃動與抖動容易造成低頻雜訊(0.01~10Hz)，請試著用強大的濾波器，將「範例肌電訊號1」的低頻雜訊濾除！



# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
from scipy.fft import fft, fftfreq, ifft #從scipy.fft函式庫中，引入頻域轉換公式fft, fftfreq, ifft
import plotly.graph_objects as go     #引入plotly.graph_objects函式庫，命名為go
import numpy as np             #引入numpy函式庫，命名為np
import pandas as pd            #引入pandas函式庫，命名為np
from google.colab import files      #從google colab函式庫中，引入files函式

# 1.先複習Python(III)-2學習的頻率能量濾除方法，把0.5Hz到40Hz的心電訊號濾除，並將訊號還原回來。

In [ ]:
#先將資料夾中的「範例心電訊號(含雜訊)」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
#來讀取剛上傳的資料吧！
df = pd.read_csv('範例心電訊號(含雜訊).txt') #以pandas的read_csv即可讀取檔案
data = np.array(df)              #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2 = data[:,0]               #指定data中的第一行資料，建立陣列data2

#稍微設定一下畫布資訊！
fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="範例心電訊號(含雜訊)",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                      #顯示圖形

In [ ]:
dots = len(data2)   #取得資料總數
period = 1/500     #宣告週期參數

xf = fftfreq(dots, period)            #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
yf = fft(data2)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，

delete_frequency_start1 = 0             #選取特定濾除頻率起點
delete_frequency_end1 =  0.5             #選取特定濾除頻率終點

delete_frequency_start2 = 40             #選取特定濾除頻率起點

for i in range(len(xf)):      #將選取範圍的頻率能量濾除(設為0)
  if abs(xf[i])>=delete_frequency_start1 and abs(xf[i])<=delete_frequency_end1:  #將選取範圍從起點(0)到終點(0.5)的頻率能量濾除
    yf[i] = 0
    print("已濾除",abs(xf[i]),"Hz的頻率能量")
  if abs(xf[i])>=delete_frequency_start2:                      #將選取範圍從起點(40)以上的頻率能量都濾除
    yf[i] = 0
    print("已濾除",abs(xf[i]),"Hz的頻率能量")

yf_half = np.abs(yf[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf_normalized = 2 / dots * yf_half        #頻域轉換後將資料正規畫

iyf = ifft(yf, n=np.size(yf))  #將濾除特定頻率後的訊號，轉換回時域訊號，須提供訊號本身與訊號點數

fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=iyf.real,                #新增一條線條，，將轉換訊號的時域部分畫出
))
fig.show()                 #顯示圖形

#3. 訊號處理有一種更強大的時域濾波器方法，直接在時域處理，就能夠快速地取得正確頻率範圍的訊號!?

In [ ]:
#強大的濾波器，不需要再轉換到頻域即可快速濾除雜訊，還能達到一樣的效果！
from scipy.signal import butter, filtfilt #從scipy.signal函式庫中，引入濾波器函式butter, filtfilt
#訊號處理常用的濾波器，可直接根據定義的頻域能量範圍，將正確的訊號過濾出來
#使用方法需要提供(時域訊號、資料取樣頻率、特定保留頻率起點、特定保留頻率終點)
def super_filter(data, frequency, save_frequency_start, save_frequency_end):   #super_filter(時域訊號, 週期, 特定保留頻率起點, 特定保留頻率終點)
  b, a = butter(3, [save_frequency_start, save_frequency_end], fs=frequency, btype='band')
  y = filtfilt(b, a, data)
  return y

In [ ]:
data3 = data[:,0]           #指定data中的第一行資料，重新建立一筆陣列data3
period = 1/500     #宣告資料取樣週期參數
frequency = 500     #宣告資料取樣頻率參數

data4 = super_filter(data3, frequency, 0.5, 40) #將時域訊號, 資料取樣頻率, 心電訊號保留頻率起點,
                           #心電訊號保留頻率終點代入強大的濾波器(super filter)

fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data4,                #新增一條線條，，將轉換訊號的時域部分畫出
))
fig.update_layout(                  #更新圖形的說明
    title="強大的時域濾波器直接處理心電訊號的時域波形",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                 #顯示圖形

In [ ]:
#回頭來觀察強大濾波器方法的頻域能量分布與原始訊號的頻域能量分布之間的差異
dots = len(data4)   #取得資料總數
period = 1/500     #宣告資料取樣週期參數

xf2 = fftfreq(dots, period)            #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
yf2 = fft(data4)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列

yf2_half = np.abs(yf2[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf2_normalized = 2 / dots * yf2_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(
    x=xf2, y=yf2_normalized, #新增一條線條在此圖形，將xf2, yf2_normalized在x, y軸上畫出
    name="強大時域濾波器方法的頻域能量分布"
))
fig.update_layout(                  #更新圖形的說明
    title="原始訊號與強大時域濾波器方法的頻域能量分布比較",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                     #顯示圖形

dots = len(data2)    #取得資料總數
period = 1/500     #宣告資料取樣週期參數
source_xf = fftfreq(dots, period)             #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
source_yf = fft(data2)                   #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列
source_yf_half = np.abs(source_yf[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
source_yf_normalized = 2 / dots * source_yf_half        #頻域轉換後將資料正規畫

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=xf2, y=source_yf_normalized, #新增一條線條在此圖形，將xf2, yf2_normalized在x, y軸上畫出
        marker=dict(        #幫第二條線用紅色標記
        color='red',
    ),
    name="原始訊號的頻域能量分布"
))

fig2.show()                     #顯示圖形

In [ ]:
#回頭來觀察強大濾波器方法的頻域能量分布與轉頻域處理方法，疊圖查看之間的差異
fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(
    x=xf2, y=yf2_normalized, #新增一條線條在此圖形，將xf2, yf2_normalized在x, y軸上畫出
    name='強大的時域濾波器方法的頻域能量分布'
))

fig.add_trace(go.Scatter(
    x=xf, y=yf_normalized  ,#新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
    marker=dict(        #幫第二條線用紅色標記
        color='red',
    ),
    name='轉頻域處理方法的頻域能量分布'
))
fig.update_layout(                  #更新圖形的說明
    title="兩種方法的頻濾能量分布",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                     #顯示圖形

In [ ]:
#此處細節應為大學範圍
#強大時域濾波器的頻率衰減量分布圖
#教師投影片講解
from scipy.signal import freqz
b, a = butter(3, [0.5, 40], fs=frequency, btype='band')
w, h = freqz(b, a, fs=frequency, worN=2000)

fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(
    x=w, y=abs(h)              #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
))
fig.update_layout(                  #更新圖形的說明
    title="強大時域濾波器的頻率衰減量分布圖",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                     #顯示圖形

#4. 應用強大的時域濾波器，拯救受到嚴重移動假影干擾的肌電訊號

> 在學術上，肌電訊號的頻率能量是從20Hz到400Hz。

> 在施力的過程，手臂或腿部的晃動與抖動容易造成低頻雜訊(0.01~10Hz)，請試著用強大的濾波器，將「範例肌電訊號1」的低頻雜訊濾除！


In [ ]:
#請自行在下方
#1.匯入「範例肌電訊號1」
#2.嘗試用plotly印出「範例肌電訊號1」
#3.利用強大的時域濾波器(super_filter)，將「範例肌電訊號1」的20Hz~400Hz的訊號濾出
#4.最後用plotly印出波形觀察

#備註：此次肌電訊號檔案中，資料的紀錄頻率是每秒1000個點，資料週期是頻率的倒數(1/1000)


